## Imports

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import datetime as datetime

from scipy import stats
from pyhive import presto
from datetime import datetime, timedelta

import warnings
warnings.filterwarnings('ignore')

In [2]:
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 300)

In [3]:
## Connection
connection = presto.connect(
        host='presto-gateway.serving.data.production.internal',
        port=80,
        protocol='http',
        catalog='hive',
        username='manoj.ravirajan@rapido.bike'
)

presto_port = '80'
username = 'manoj.ravirajan@rapido.bike'
conn1 = presto.connect('bi-presto.serving.data.production.internal', presto_port, username)
conn2 = presto.connect('bi-trino-2.serving.data.production.internal', presto_port, username)
conn3= presto.connect('processing-2.processing.data.production.internal',presto_port,username)
conn4= presto.connect('processing.processing.data.production.internal',presto_port,username)

## Datasets & Parameter

In [29]:
## base

base_query = f"""

    with base as (
        select 
            customer_id,
            segment,
            case 
            when office_orders = '0' or office_orders is null then 'other' 
            when office_orders > '0' then 'office'
            else 'other' end office_tag
            
        from 
            hive.experiments_internal.customer_base_office_transit_tag_amj2025
        
        ),
        
        
        os as (
        
        select 
            user_id,
            min(platform) os
        from 
            clevertap.clevertap_customer_order_activity
        where 
            yyyymmdd >= '20250217'
            and yyyymmdd <= '20250316'
            and current_city = 'Bangalore'
            and platform != ''
            and platform is not null
        group by 1
        ),
        
        
        app_category as (
        
        select 
            userid,
            case when travelRidehailing = 1 then 'yes' else 'no' end travelRidehailing,
            case when deliveryFood > 0 or deliveryGrocery > 0 then 'yes' else 'no' end quick_commerce
        from 
            reports.sql_ingestion_customer_appography_category_view
        where 
            yyyymmdd = '20250406'
        )
        
        

        select
            base.*,
            case 
            when os in ('Android', 'iOS') then os
            when os is null and userid is not null then 'Android'
            else 'Blank'
            end os,
            -- case when userid is null then 'iOS' else 'android' end os,
            case when userid is null then 'blank' else app_category.travelRidehailing end travelRidehailing,
            case when userid is null then 'blank' else app_category.quick_commerce end quick_commerce
    
        from
            base
        left join 
            app_category
            on base.customer_id = app_category.userid
        left join 
            os
            on base.customer_id = os.user_id
"""

df_base_query = pd.read_sql(base_query, connection)
df_base_query

,customer_id,segment,office_tag,os,travelRidehailing,quick_commerce
0,62988ce74894f9ba3f2fade1,4-OTHER,other,Android,yes,no
1,6416901120601d6a8930eb0f,3-MONTHLY,office,Android,yes,yes
2,64a270e171a6b445e3d30e7b,4-OTHER,other,Android,no,yes
3,65196f1fecdf8d1de09d2d5e,3-MONTHLY,office,Android,yes,no
4,66b5be6e54620ceddf057553,5-INACTIVE,other,Android,no,no
...,...,...,...,...,...,...
4929720,6610a1d7469c8ff701bc6872,4-OTHER,other,iOS,blank,blank
4929721,65e20fb4a743e0266c31d455,1-DAILY,other,iOS,blank,blank
4929722,619d24623f25c305e5e5d0a0,3-MONTHLY,other,Android,yes,yes
4929723,62e610b0506bfd21e07a71d7,3-MONTHLY,office,Android,yes,yes


In [31]:
df_base_query.shape

(4929725, 6)

In [32]:
df_base_query.to_csv('rha_qc_usecase_segment_gcs.csv', index=False, header=False)

In [7]:
test1 = pd.read_csv('rha_qc_usecase_segment.csv')

In [8]:
test1.shape

(4929725, 6)

In [14]:
## base

base_query1 = f"""

    with orders as (

    select
        yyyymmdd,
        customer_id,
        service_obj_service_name service_name,
        order_id,
        modified_order_status,
        bid_type,
        pickup_location_selection_mode psm,
        customer_set_pickup_ride_started_distance_meters haps,
        customer_cancelled_epoch,
        order_requested_epoch
    from 
        orders.order_logs_fact
    where 
        yyyymmdd >= '202502017'
        and yyyymmdd <= '20250316'
        and city_name = 'Bangalore'
        and service_obj_service_name in ('Auto','Auto Lite','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
        -- and customer_id = '6288c6694281e53bbe33d133'
)


select 
    customer_id,
    count(distinct order_id) rr_count,
    count(distinct case when modified_order_status = 'dropped' then order_id end) net_count,
    count(distinct case when modified_order_status = 'COBRM' then order_id end) cobrm_count,
    count(distinct case when modified_order_status = 'COBRA' then order_id end) cobra_count,
    count(distinct case when modified_order_status = 'OCARA' then order_id end) ocara_count,
    count(distinct case when modified_order_status in ('COBRA', 'COBRM', 'OCARA') then order_id end) canceltn_count,
    
    count(distinct case when modified_order_status = 'COBRA' and ((cast(customer_cancelled_epoch as bigint) - cast(order_requested_epoch as bigint))/1000) <= 30 then order_id end) quick_cobra,
    count(distinct case when modified_order_status = 'OCARA' and ((cast(customer_cancelled_epoch as bigint) - cast(order_requested_epoch as bigint))/1000) <= 30 then order_id end) quick_ocara,
    
    avg(distinct case when modified_order_status = 'COBRA' then ((cast(customer_cancelled_epoch as bigint) - cast(order_requested_epoch as bigint))/1000) end) cobra_ttc,
    avg(distinct case when modified_order_status = 'OCARA' then ((cast(customer_cancelled_epoch as bigint) - cast(order_requested_epoch as bigint))/1000) end) ocara_ttc,
    
    count(distinct case when modified_order_status = 'expired' then order_id end) expired_count,
    count(distinct case when modified_order_status = 'aborted' then order_id end) aborted_count,
    count(distinct case when haps <= 30 and modified_order_status = 'dropped' then order_id end) haps_orders,
    count(distinct case when bid_type in ('positive bid', 'negative bid') then order_id end) bid_orders,
    count(distinct case when bid_type = 'negative bid' then order_id end) negative_bid,
    count(distinct case when bid_type = 'positive bid' then order_id end) positive_bid,
    
    count(distinct case when psm = 'gps' then order_id end) gross_psm_gps,
    count(distinct case when psm in ('favourites', 'recent_history') then order_id end) gross_psm_fav_recent_history,
    count(distinct case when psm = 'google_search' then order_id end) gross_psm_google_search,
    count(distinct case when psm = 'stickyLocation' then order_id end) gross_psm_stickyLocation,
    count(distinct case when psm = 'hotspot' then order_id end) gross_psm_hotspot,
    
    count(distinct case when modified_order_status = 'dropped' and psm = 'gps' then order_id end) net_psm_gps,
    count(distinct case when modified_order_status = 'dropped' and psm in ('favourites', 'recent_history') then order_id end) net_psm_fav_recent_history,
    count(distinct case when modified_order_status = 'dropped' and psm = 'google_search' then order_id end) net_psm_google_search,
    count(distinct case when modified_order_status = 'dropped' and psm = 'stickyLocation' then order_id end) net_psm_fav_stickyLocation,
    count(distinct case when modified_order_status = 'dropped' and psm = 'hotspot' then order_id end) net_psm_hotspot
    
from 
    orders
group by 1
"""

df_base_query1 = pd.read_sql(base_query1, conn3)
df_base_query1

,customer_id,rr_count,net_count,cobrm_count,cobra_count,ocara_count,canceltn_count,quick_cobra,quick_ocara,cobra_ttc,ocara_ttc,expired_count,aborted_count,haps_orders,bid_orders,negative_bid,positive_bid,gross_psm_gps,gross_psm_fav_recent_history,gross_psm_google_search,gross_psm_stickyLocation,gross_psm_hotspot,net_psm_gps,net_psm_fav_recent_history,net_psm_google_search,net_psm_fav_stickyLocation,net_psm_hotspot
0,5fd60aa2ecbb3f5cb7267aef,20,19,0,0,0,0,0,0,NaN,NaN,1,0,19,19,19,0,1,0,0,18,0,1,0,0,17,0
1,662f04138659bbe66d002d1f,7,5,0,0,2,2,0,0,NaN,422.500000,0,0,5,1,0,1,5,0,1,1,0,4,0,0,1,0
2,64c91b517efde5cd7a26fdb6,8,4,0,1,3,4,0,0,68.0,127.666667,0,0,4,0,0,0,0,1,0,3,4,0,0,0,3,1
3,644b6c478fac8a146bc254e7,33,22,0,7,4,11,1,1,77.0,400.750000,0,0,19,9,9,0,0,0,1,6,6,0,0,1,3,5
4,60deb3de09aa99f0e93dcf32,3,2,0,0,1,1,0,0,NaN,303.000000,0,0,2,0,0,0,0,2,0,0,0,0,2,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3907803,6543a42e1e00966cb36d5d7a,1,1,0,0,0,0,0,0,NaN,NaN,0,0,1,0,0,0,0,0,1,0,0,0,0,1,0,0
3907804,606347f6193b5c54f704c482,1,1,0,0,0,0,0,0,NaN,NaN,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0
3907805,5cc5bfa154bc7263ff54c759,1,0,0,0,1,1,0,0,NaN,599.000000,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0
3907806,6766dbec51c8abcc7502b6b6,1,0,0,0,1,1,0,0,NaN,641.000000,0,0,0,1,0,1,1,0,0,0,0,0,0,0,0,0


In [15]:
df_base_query1

,customer_id,rr_count,net_count,cobrm_count,cobra_count,ocara_count,canceltn_count,quick_cobra,quick_ocara,cobra_ttc,ocara_ttc,expired_count,aborted_count,haps_orders,bid_orders,negative_bid,positive_bid,gross_psm_gps,gross_psm_fav_recent_history,gross_psm_google_search,gross_psm_stickyLocation,gross_psm_hotspot,net_psm_gps,net_psm_fav_recent_history,net_psm_google_search,net_psm_fav_stickyLocation,net_psm_hotspot
0,5fd60aa2ecbb3f5cb7267aef,20,19,0,0,0,0,0,0,NaN,NaN,1,0,19,19,19,0,1,0,0,18,0,1,0,0,17,0
1,662f04138659bbe66d002d1f,7,5,0,0,2,2,0,0,NaN,422.500000,0,0,5,1,0,1,5,0,1,1,0,4,0,0,1,0
2,64c91b517efde5cd7a26fdb6,8,4,0,1,3,4,0,0,68.0,127.666667,0,0,4,0,0,0,0,1,0,3,4,0,0,0,3,1
3,644b6c478fac8a146bc254e7,33,22,0,7,4,11,1,1,77.0,400.750000,0,0,19,9,9,0,0,0,1,6,6,0,0,1,3,5
4,60deb3de09aa99f0e93dcf32,3,2,0,0,1,1,0,0,NaN,303.000000,0,0,2,0,0,0,0,2,0,0,0,0,2,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3907803,6543a42e1e00966cb36d5d7a,1,1,0,0,0,0,0,0,NaN,NaN,0,0,1,0,0,0,0,0,1,0,0,0,0,1,0,0
3907804,606347f6193b5c54f704c482,1,1,0,0,0,0,0,0,NaN,NaN,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0
3907805,5cc5bfa154bc7263ff54c759,1,0,0,0,1,1,0,0,NaN,599.000000,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0
3907806,6766dbec51c8abcc7502b6b6,1,0,0,0,1,1,0,0,NaN,641.000000,0,0,0,1,0,1,1,0,0,0,0,0,0,0,0,0


In [16]:
df_base_query1.to_csv('olf_g2n_30days_20250317_v1.csv', index=False, header=False)

In [18]:
## base

base_query_v2 = f"""

with ao as (

    select 
        yyyymmdd,
        platform,
        user_id,
        CAST(CAST(ct_session_id AS DECIMAL) AS VARCHAR) || ' - ' || phone AS ao_session_id,
        min(epoch) AS ao_epoch
    from 
        clevertap.clevertap_customer_order_activity
    where 
        yyyymmdd >= '20250401'
        and yyyymmdd <= '20250413'
        and current_city = 'Bangalore'
        and serviceable = 'true'
        and order_activity_source = 'appOpen'
    group by 1,2,3,4

    ),
    
    fe as (
    
    select 
        yyyymmdd,
        user_id,
        CAST(CAST(ct_session_id AS DECIMAL) AS VARCHAR) || ' - ' || phone AS fe_session_id,
        min(epoch) AS fe_epoch
    from 
        canonical.clevertap_customer_fare_estimate
    where 
        yyyymmdd >= '20250401'
        and yyyymmdd <= '20250413'
        and current_city = 'Bangalore'
    group by 1,2,3
    ),
    
    ao_fe as (
    
    select
        ao.*,
        fe.fe_epoch,
        fe.fe_session_id,
        ((fe.fe_epoch - ao.ao_epoch)/1000) ao2fe
    from 
        ao
    left join 
        fe
        on ao.yyyymmdd = fe.yyyymmdd
        and ao.user_id = fe.user_id
        and ao.ao_session_id = fe.fe_session_id
    )
    
    -- select
    --     *
    -- from 
    --     ao_fe
    
    select
        yyyymmdd,
        platform,
        count(distinct ao_session_id) ao_session,
        count(distinct fe_session_id) fe_session,
        try(count(distinct fe_session_id)*100.00/count(distinct ao_session_id)) ao2fe
    from 
        ao_fe
    group by 1,2
    order by 1,2



"""

df_base_query2 = pd.read_sql(base_query_v2, conn4)
df_base_query2

,yyyymmdd,platform,ao_session,fe_session,ao2fe
0,20250401,Android,785229,642264,81.79
1,20250401,iOS,221362,175492,79.28
2,20250402,Android,754822,609359,80.73
3,20250402,iOS,218253,171519,78.59
4,20250403,Android,758004,603292,79.59
5,20250403,iOS,219603,170660,77.71
6,20250404,Android,705089,562064,79.72
7,20250404,iOS,204075,156781,76.83
8,20250405,Android,729569,582910,79.90
9,20250405,iOS,202290,154710,76.48


In [19]:
df_base_query2.to_clipboard(index=False)

In [22]:
df_control_v1 = pd.read_csv('/Users/E2074/local-datasets/customer/loyalty/experiment1/final/control_group_t1.csv')
df_control_v2 = pd.read_csv('/Users/E2074/local-datasets/customer/loyalty/experiment1/final/control_group_t2.csv')
df_control_v3 = pd.read_csv('/Users/E2074/local-datasets/customer/loyalty/experiment1/final/control_group_t3.csv')

In [26]:
display(df_control_v1.shape)
display(df_control_v2.shape)
display(df_control_v3.shape)

(150000, 1)

(150000, 1)

(149997, 1)

In [27]:
result = pd.concat([df_control_v1, df_control_v2, df_control_v3], ignore_index=True)
result

,customer_mobile
0,6202742044
1,7975903747
2,7845811202
3,9019388700
4,7047699844
...,...
449992,9994397130
449993,8867345603
449994,9914646655
449995,7848999824


In [34]:
result.to_csv('/Users/E2074/local-datasets/customer/loyalty/experiment1/final/control_group.csv', header=False, index=False)

In [33]:
result

,customer_mobile
0,6202742044
1,7975903747
2,7845811202
3,9019388700
4,7047699844
...,...
449992,9994397130
449993,8867345603
449994,9914646655
449995,7848999824


In [4]:
## base

base_query = f"""

with rr_session as ( 
        select
            user_id,
            count(distinct CAST(CAST(ct_session_id AS DECIMAL) AS VARCHAR) || ' - ' || phone) customer_rr_sessions
        from 
            canonical.clevertap_customer_request_rapido
        where
                yyyymmdd >= '20250217'
                and yyyymmdd <= '20250316'
                and current_city = 'Bangalore'
                -- and user_id = '6288c6694281e53bbe33d133'
        group by 1
    )
    
    select
        *
    from 
        rr_session
"""
df_base_query = pd.read_sql(base_query, connection)
df_base_query

,user_id,customer_rr_sessions
0,5b03870d4c87115d44d28fde,6
1,66b4b74960e409541eddfb37,1
2,66741ffc26174a06a2332874,35
3,674067064036528e0a5e7ce4,6
4,65d0b6a119d22e76c59805a9,14
...,...,...
3147680,6575ebf799e7a114c5453192,2
3147681,63e519301f53877dac1372ec,1
3147682,5d03bad9abbf48476381e372,1
3147683,5cfabd65ca6e29211175fa43,1


In [5]:
df_base_query

,user_id,customer_rr_sessions
0,5b03870d4c87115d44d28fde,6
1,66b4b74960e409541eddfb37,1
2,66741ffc26174a06a2332874,35
3,674067064036528e0a5e7ce4,6
4,65d0b6a119d22e76c59805a9,14
...,...,...
3147680,6575ebf799e7a114c5453192,2
3147681,63e519301f53877dac1372ec,1
3147682,5d03bad9abbf48476381e372,1
3147683,5cfabd65ca6e29211175fa43,1


In [6]:
df_base_query.to_csv('customer_rr_sessions.csv', header=False, index=False)